# 02 ASR quality (Gate 3, F2)

Runs the functions of `src/asr/quality.py` on the manifests and the ASR tag from `configs/asr.yaml`. Needs the transcripts from `python -m src.asr.transcribe --split all`.

In [ ]:
# parameters: MANIFEST_DIR = None uses data/manifests; the smoke run sets data/processed/smoke_manifests
MANIFEST_DIR = None

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
from src.asr import quality
from src.common import load_yaml
TAG = load_yaml('configs/asr.yaml')['tag']
SMOKE = quality.is_smoke(MANIFEST_DIR)
paths = quality.default_paths(TAG, SMOKE)
TAG, SMOKE

In [ ]:
df = quality.load_rows(TAG, MANIFEST_DIR)
df[['utt_id', 'split', 'emotion', 'gold_text', 'asr_text', 'wer']].head()

## WER per split

In [ ]:
(quality.wer_per_split(df).style.format({'WER': '{:.1%}', 'WER_policy_kept': '{:.1%}'}))

## WER per split x class

In [ ]:
pc = quality.wer_per_class(df)
pc.style.format({'train': '{:.1%}', 'dev': '{:.1%}', 'test': '{:.1%}', 'mean_dur_s': '{:.2f}', 'mean_gold_words': '{:.1f}'}, na_rep='')

In [ ]:
quality.plot_wer_per_class(pc, TAG, paths['fig'])
display(Image(filename=str(paths['fig'])))

## Length ratio, hallucination flags, duration buckets

In [ ]:
display(quality.ratio_stats(df)); display(quality.flag_counts(df))
quality.wer_by_duration(df).style.format({'train': '{:.1%}', 'dev': '{:.1%}', 'test': '{:.1%}', 'all': '{:.1%}'}, na_rep='')

## 15 highest-WER rows (manual plausibility check)

In [ ]:
pd.set_option('display.max_colwidth', 120)
quality.top_wer_rows(df)